# Подключаемые модальности: от CLIP до входного и выходного порта

Вопрос лаборатории: можно ли добавить изображение и звук к замороженному ядру, затем снять звук и получить в точности то же текстовое ядро? Ожидаемые инварианты: ручная ошибка CLIP равна `0.126928`, правильные пары лучше перемешанных, параметры ядра и внешнего декодера не меняются, а удаление порта не влияет на оставшиеся порты. Всё выполняется на CPU без сети. Основная часть написана на NumPy; только раздел 7 загружает полный реестр портов из главы и требует PyTorch.

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np

rng = np.random.default_rng(17)
print('NumPy:', np.__version__)


## 1. CLIP руками

Нормируем два набора векторов, строим все попарные сходства и усредняем две перекрёстные энтропии. Пакет сам предоставляет отрицательные пары.

In [ ]:
def logsumexp(x, axis):
    maximum = np.max(x, axis=axis, keepdims=True)
    return np.squeeze(maximum + np.log(np.exp(x - maximum).sum(axis=axis, keepdims=True)), axis=axis)

def clip_loss(image_features, text_features, temperature=0.1):
    image_features = image_features / np.linalg.norm(image_features, axis=-1, keepdims=True)
    text_features = text_features / np.linalg.norm(text_features, axis=-1, keepdims=True)
    logits = image_features @ text_features.T / temperature
    diagonal = np.arange(logits.shape[0])
    image_to_text = (-logits[diagonal, diagonal] + logsumexp(logits, axis=1)).mean()
    text_to_image = (-logits[diagonal, diagonal] + logsumexp(logits, axis=0)).mean()
    return 0.5 * (image_to_text + text_to_image), logits

image_vectors = np.array([[1.0, 0.0], [0.0, 1.0]])
text_vectors = np.array([[0.8, 0.6], [0.6, 0.8]])
manual_clip_loss, similarity_logits = clip_loss(image_vectors, text_vectors)
np.testing.assert_allclose(similarity_logits, [[8.0, 6.0], [6.0, 8.0]], atol=1e-12)
np.testing.assert_allclose(manual_clip_loss, 0.12692801104297224, atol=1e-12)
print('CLIP loss:', manual_clip_loss)


## 2. Два физических сигнала — один контракт

Изображение `[B, 4, 4]` делим на четыре участка `2×2`. Звук `[B, 8]` делим на четыре окна по два отсчёта. Оба кодировщика возвращают `[B, 4, 2]`, но два числа имеют разный физический смысл.

In [ ]:
def image_encoder(images):
    assert images.ndim == 3 and images.shape[1:] == (4, 4)
    patches = images.reshape(images.shape[0], 2, 2, 2, 2).transpose(0, 1, 3, 2, 4)
    patches = patches.reshape(images.shape[0], 4, 4)
    return np.stack([patches.mean(axis=-1), patches.std(axis=-1)], axis=-1)

def audio_encoder(waves):
    assert waves.ndim == 2 and waves.shape[1] == 8
    frames = waves.reshape(waves.shape[0], 4, 2)
    return np.stack([frames.mean(axis=-1), frames[:, :, 1] - frames[:, :, 0]], axis=-1)

probe_images = rng.normal(size=(3, 4, 4))
probe_waves = rng.normal(size=(3, 8))
assert image_encoder(probe_images).shape == (3, 4, 2)
assert audio_encoder(probe_waves).shape == (3, 4, 2)


## 3. Замороженное языковое ядро

Для быстрого опыта ядро только усредняет входные позиции и применяет неизменную выходную матрицу. Это не полноценный трансформер, но граница порта та же: на входе `[B, K, d_lm]`, на выходе текстовые логиты. Контрольная сумма позволит доказать отсутствие скрытого обновления.

In [ ]:
D_LM = 4
CORE_HEAD = np.array([[3.0, -3.0], [-3.0, 3.0], [0.0, 0.0], [0.0, 0.0]])
TEXT_TABLE = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.5, 0.5, 0.0, 0.0]])

def digest(*arrays):
    state = hashlib.sha256()
    for array in arrays:
        state.update(np.asarray(array).tobytes())
    return state.hexdigest()

def core_logits(tokens):
    assert tokens.ndim == 3 and tokens.shape[-1] == D_LM
    return tokens.mean(axis=1) @ CORE_HEAD

core_sha_before = digest(CORE_HEAD, TEXT_TABLE)
text_probe_before = core_logits(TEXT_TABLE[[0, 2]][None, :, :]).copy()


## 4. Учим только два соединителя

Синтетический класс 0 имеет положительный средний сигнал, класс 1 — отрицательный. Проектор `2→4` обучается через замороженную выходную матрицу. Функция возвращает все промежуточные формы, поэтому границу можно проверить независимо.

In [ ]:
labels = np.tile(np.array([0, 1]), 32)
sign = np.where(labels == 0, 1.0, -1.0)
images = sign[:, None, None] + 0.10 * rng.normal(size=(64, 4, 4))
waves = sign[:, None] + 0.10 * rng.normal(size=(64, 8))

def softmax(logits):
    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)

def evaluate_port(raw, labels, encoder, weight, bias):
    features = encoder(raw)
    tokens = features @ weight + bias
    logits = core_logits(tokens)
    probabilities = softmax(logits)
    loss = -np.log(probabilities[np.arange(labels.size), labels] + 1e-12).mean()
    return loss, logits, features, tokens

def train_port(raw, labels, encoder, steps=180, learning_rate=0.2):
    features = encoder(raw)
    weight = rng.normal(scale=0.05, size=(features.shape[-1], D_LM))
    bias = np.zeros(D_LM)
    history = []
    for _ in range(steps):
        tokens = features @ weight + bias
        logits = core_logits(tokens)
        probabilities = softmax(logits)
        loss = -np.log(probabilities[np.arange(labels.size), labels] + 1e-12).mean()
        history.append(loss)

        d_logits = probabilities
        d_logits[np.arange(labels.size), labels] -= 1.0
        d_logits /= labels.size
        d_pooled = d_logits @ CORE_HEAD.T
        d_tokens = np.repeat(d_pooled[:, None, :], features.shape[1], axis=1) / features.shape[1]
        d_weight = np.einsum('bsd,bse->de', features, d_tokens)
        d_bias = d_tokens.sum(axis=(0, 1))
        weight -= learning_rate * d_weight
        bias -= learning_rate * d_bias
    return weight, bias, np.array(history)


In [ ]:
image_weight, image_bias, image_history = train_port(images, labels, image_encoder)
audio_weight, audio_bias, audio_history = train_port(waves, labels, audio_encoder)
assert image_history[-1] < image_history[0] * 0.02
assert audio_history[-1] < audio_history[0] * 0.02

image_loss, image_logits, image_features, image_tokens = evaluate_port(
    images, labels, image_encoder, image_weight, image_bias)
audio_loss, audio_logits, audio_features, audio_tokens = evaluate_port(
    waves, labels, audio_encoder, audio_weight, audio_bias)
assert image_features.shape == audio_features.shape == (64, 4, 2)
assert image_tokens.shape == audio_tokens.shape == (64, 4, D_LM)
assert (image_logits.argmax(axis=-1) == labels).mean() == 1.0
assert (audio_logits.argmax(axis=-1) == labels).mean() == 1.0
print('final losses:', image_loss, audio_loss)


## 5. Проверяем, что модель не угадывает по частоте класса

Сдвиг на один элемент меняет каждый сигнал на сигнал противоположного класса. Если ошибка не выросла, порт или ядро игнорирует модальность.

In [ ]:
shuffled_image_loss, _, _, _ = evaluate_port(
    np.roll(images, 1, axis=0), labels, image_encoder, image_weight, image_bias)
shuffled_audio_loss, _, _, _ = evaluate_port(
    np.roll(waves, 1, axis=0), labels, audio_encoder, audio_weight, audio_bias)
assert shuffled_image_loss > image_loss + 1.0
assert shuffled_audio_loss > audio_loss + 1.0
modality_reliance = {
    'image': shuffled_image_loss - image_loss,
    'audio': shuffled_audio_loss - audio_loss,
}
print('reliance margins:', modality_reliance)


## 6. Реестр: снять звук и добавить глубину

Порт владеет кодировщиком и проектором, но не ядром. После удаления звука логиты изображения и контрольная сумма ядра обязаны остаться прежними. Новый датчик глубины подключается тем же контрактом.

In [ ]:
registry = {
    'image': {'encoder': image_encoder, 'weight': image_weight, 'bias': image_bias},
    'audio': {'encoder': audio_encoder, 'weight': audio_weight, 'bias': audio_bias},
}
image_logits_before_detach = image_logits.copy()
detached_audio = registry.pop('audio')
assert set(registry) == {'image'}
image_loss_after, image_logits_after, _, _ = evaluate_port(
    images, labels, registry['image']['encoder'], registry['image']['weight'], registry['image']['bias'])
np.testing.assert_array_equal(image_logits_after, image_logits_before_detach)
assert digest(CORE_HEAD, TEXT_TABLE) == core_sha_before
np.testing.assert_array_equal(core_logits(TEXT_TABLE[[0, 2]][None, :, :]), text_probe_before)

def depth_encoder(depth_maps):
    features = image_encoder(depth_maps)
    return np.stack([features[:, :, 0], np.abs(features[:, :, 0])], axis=-1)

depth_maps = sign[:, None, None] + 0.08 * rng.normal(size=(64, 4, 4))
depth_weight, depth_bias, depth_history = train_port(depth_maps, labels, depth_encoder)
registry['depth'] = {'encoder': depth_encoder, 'weight': depth_weight, 'bias': depth_bias}
depth_loss, depth_logits, _, depth_tokens = evaluate_port(
    depth_maps, labels, depth_encoder, depth_weight, depth_bias)
assert depth_tokens.shape == (64, 4, D_LM)
assert (depth_logits.argmax(axis=-1) == labels).mean() == 1.0
assert depth_history[-1] < depth_history[0] * 0.02


## 7. Полный реестр портов из главы (PyTorch)

В разделе 6.5 «Порт, который можно добавить и снять» книга показывает сокращённый листинг. Полная версия лежит в `labs/ch06/pluggable_port_registry.py`: класс `InputPort` (кодировщик, проектор и два граничных вектора — начало и конец сигнала) и класс `PortRegistry` с методами `attach`, `detach` и `encode`.

Загружаем файл по явному пути через `importlib` и запускаем его собственную проверку `main()`. Она требует трёх вещей: второй порт с тем же именем отклоняется; четыре участка изображения дают 4 + 2 позиции; логиты текста после снятия порта совпадают с исходными бит в бит. Потом подключаем сразу два порта и снимаем звуковой. Выход порта изображения при этом не меняется, а среди параметров реестра нет ни одного параметра ядра.

In [ ]:
import importlib.util

import torch

for base in (Path.cwd(), *Path.cwd().parents):
    LAB_FILE = base / 'labs' / 'ch06' / 'pluggable_port_registry.py'
    if LAB_FILE.is_file():
        break
else:
    raise FileNotFoundError('labs/ch06/pluggable_port_registry.py: запустите ноутбук из репозитория книги')

spec = importlib.util.spec_from_file_location('ch06_pluggable_port_registry', LAB_FILE)
port_lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_lab)
print('PyTorch:', torch.__version__)

port_lab.main()   # attach -> encode -> detach, текстовые логиты бит в бит

torch.manual_seed(0)
torch_core = torch.nn.Sequential(torch.nn.Embedding(50, 8), torch.nn.Linear(8, 50))
torch_registry = port_lab.PortRegistry()
torch_registry.attach('img', port_lab.InputPort(torch.nn.Linear(12, 6), 6, 8))
torch_registry.attach('audio', port_lab.InputPort(torch.nn.Linear(2, 6), 6, 8))
core_ids = {id(t) for t in torch_core.parameters()}
assert not core_ids & {id(t) for t in torch_registry.parameters()}

img_raw = torch.randn(2, 4, 12)
with torch.no_grad():
    img_before = torch_registry.encode('img', img_raw)
detached_torch_audio = torch_registry.detach('audio')
with torch.no_grad():
    img_after = torch_registry.encode('img', img_raw)
assert list(torch_registry.ports) == ['img']
assert img_after.shape == (2, 4 + 2, 8)
assert torch.equal(img_before, img_after)
print('реестр PyTorch: звук снят, порт изображения и ядро не изменились')

## 8. Выходной порт в стиле NExT-GPT

Скрытое состояние сигнального токена имеет ширину LLM. Обучаемый проектор переводит его в условие замороженного декодера. Декодер здесь — маленькая матрица, но проверка владения параметрами совпадает с внешним генератором изображения или звука.

In [ ]:
signal_hidden = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [0.5, 0.2, 0.0, 0.0],
])
FROZEN_DECODER = np.array([[1.0, 0.5, 0.0, -0.5], [0.0, 0.5, 1.0, 0.5]])
true_condition = signal_hidden[:, :2]
target_signal = true_condition @ FROZEN_DECODER
decoder_sha_before = digest(FROZEN_DECODER)
output_weight = rng.normal(scale=0.1, size=(D_LM, 2))
output_history = []
for _ in range(300):
    condition = signal_hidden @ output_weight
    prediction = condition @ FROZEN_DECODER
    error = prediction - target_signal
    loss = np.mean(error ** 2)
    output_history.append(loss)
    d_prediction = 2.0 * error / error.size
    d_condition = d_prediction @ FROZEN_DECODER.T
    d_weight = signal_hidden.T @ d_condition
    output_weight -= 0.15 * d_weight

assert output_history[-1] < output_history[0] * 1e-4
assert digest(FROZEN_DECODER) == decoder_sha_before
print('output reconstruction loss:', output_history[-1])


## 9. Намеренно неправильный порт

Проектор ширины `2→3` выполняет матричное умножение, но нарушает контракт `d_lm=4`. Ошибка должна обнаруживаться до запуска ядра, а не проявляться как загадочное качество.

In [ ]:
wrong_weight = np.zeros((2, 3))
wrong_tokens = image_features @ wrong_weight
try:
    assert wrong_tokens.shape[-1] == D_LM, (wrong_tokens.shape, D_LM)
except AssertionError as error:
    diagnosed_shape_error = str(error)
else:
    raise AssertionError('broken port was not detected')
assert diagnosed_shape_error


## 10. Манифест дочернего артефакта

Большие веса не сохраняются в Git. Манифест фиксирует родителя, порты, контрольные суммы, формы и измеренные инварианты.

In [ ]:
manifest = {
    'id': 'booklm-multimodal-ports-spec-v1',
    'parents': {'booklm-base-init': core_sha_before},   # учебное ядро стоит за booklm-base-init
    'created_by': 'notebooks/06_pluggable_modalities.ipynb',
    'format': 'booklm-modality-port-spec',
    'format_version': 1,
    'core_sha256': core_sha_before,
    'ports': {
        'image': {'input_shape': ['B', 4, 4], 'token_shape': ['B', 4, D_LM],
                  'sha256': digest(image_weight, image_bias)},
        'depth': {'input_shape': ['B', 4, 4], 'token_shape': ['B', 4, D_LM],
                  'sha256': digest(depth_weight, depth_bias)},
    },
    'detached_ports': ['audio'],
    'environment': {'numpy': np.__version__, 'device': 'cpu'},
    'metrics': {
        'clip_loss': float(manual_clip_loss),
        'image_reliance_margin': float(modality_reliance['image']),
        'audio_reliance_margin_before_detach': float(modality_reliance['audio']),
        'output_mse': float(output_history[-1]),
        'text_core_exact_after_detach': True,
    },
}
manifest_path = Path('/tmp/booklm-multimodal-ports-spec-v1.json')
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
loaded_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert loaded_manifest['core_sha256'] == core_sha_before
assert loaded_manifest['detached_ports'] == ['audio']
assert set(loaded_manifest['ports']) == {'image', 'depth'}
print(manifest_path)


## 11. Что соответствует промышленной системе

`image_encoder` и `audio_encoder` соответствуют отдельным предобученным кодировщикам; `weight` и `bias` — соединителю LLaVA-подобного входного порта; `CORE_HEAD` — замороженному языковому ядру; `signal_hidden` — скрытым состояниям сигнальных токенов NExT-GPT; `output_weight` — выходному соединителю; `FROZEN_DECODER` — внешнему генератору.

Самостоятельная работа: замените усредняющее ядро причинным вниманием, добавьте граничные векторы и маску ошибки `-100`. Затем реализуйте порт с нулевыми воротами перекрёстного внимания и докажите точное совпадение текстовых логитов до обучения.